# Assignment 2 — EDA, Hypothesis Testing and Feature Engineering Reflection

**Course:** Data Science (Optativa I) — FURB 2026  
**Due date:** May 20, 2026  
**Dataset:** OECD GDP per Capita × Unemployment Rate (quarterly, 2025, 41 countries)

This notebook documents the full analytical workflow for Assignment 2. It follows the required tasks in order: data overview, descriptive statistics, distribution and dispersion analysis, correlation analysis, and hypothesis testing. Each section includes explanatory text to justify methodological choices and interpret results.

---
## 1. Setup and Data Loading

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Load the merged analytical dataset (semicolon separator — OECD export convention)
df = pd.read_csv("datasets/gdp_unemployment_merged.csv", sep=";")

# Establish quarter order for consistent sorting and display
quarter_order = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]
df["QUARTER"] = pd.Categorical(df["QUARTER"], categories=quarter_order, ordered=True)
df = df.sort_values(["REF_AREA", "QUARTER"]).reset_index(drop=True)

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(8)

Dataset loaded: 146 rows × 6 columns


,REF_AREA,QUARTER,GDP_PER_CAPITA_USD_PPP,OBS_STATUS,REF_YEAR_PRICE,UNE_RATE_PCT
0,AUS,2025Q1,57487.7,A,2020.0,4.05
1,AUS,2025Q2,57802.9,A,2020.0,4.17
2,AUS,2025Q3,57839.5,A,2020.0,4.32
3,AUT,2025Q1,62125.2,A,2020.0,5.45
4,AUT,2025Q2,62144.6,A,2020.0,5.73
5,AUT,2025Q3,62293.0,A,2020.0,5.73
6,AUT,2025Q4,62241.4,A,2020.0,5.83
7,BEL,2025Q1,62261.3,P,2020.0,6.20


---
## 2. Data Overview

Before any analysis, we inspect the structure, types, and completeness of the dataset. This step verifies that the cleaning pipeline (Unit 2) produced the expected output and identifies any residual issues.

In [ ]:
# Shape, column types, and missing values
print("=== Shape ===")
print(df.shape)

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum())

=== Shape ===
(146, 6)

=== Data Types ===
REF_AREA                       str
QUARTER                   category
GDP_PER_CAPITA_USD_PPP     float64
OBS_STATUS                     str
REF_YEAR_PRICE             float64
UNE_RATE_PCT               float64
dtype: object

=== Missing Values ===
REF_AREA                  0
QUARTER                   0
GDP_PER_CAPITA_USD_PPP    0
OBS_STATUS                0
REF_YEAR_PRICE            0
UNE_RATE_PCT              0
dtype: int64


In [ ]:
# Observations per quarter — confirms temporal coverage
print("=== Observations per Quarter ===")
print(df["QUARTER"].value_counts().sort_index())

# OBS_STATUS distribution — how many records are final vs provisional vs estimated
print("\n=== GDP Observation Status (OBS_STATUS) ===")
print(df["OBS_STATUS"].value_counts())
print("  A = authoritative (final), P = provisional, E = estimated")

# Number of distinct countries in the dataset
print(f"\n=== Countries in Dataset ===")
print(f"Total unique countries: {df['REF_AREA'].nunique()}")
print(df["REF_AREA"].value_counts().sort_index())

=== Observations per Quarter ===
QUARTER
2025Q1    38
2025Q2    38
2025Q3    38
2025Q4    32
Name: count, dtype: int64

=== GDP Observation Status (OBS_STATUS) ===
OBS_STATUS
A    81
P    58
E     7
Name: count, dtype: int64
  A = authoritative (final), P = provisional, E = estimated

=== Countries in Dataset ===
Total unique countries: 38
REF_AREA
AUS    3
AUT    4
BEL    4
BGR    4
CAN    4
CHL    4
COL    3
CRI    4
CZE    4
DEU    4
DNK    4
ESP    4
EST    4
FIN    4
FRA    4
GBR    4
GRC    4
HRV    4
HUN    4
IRL    4
ISL    4
ISR    3
ITA    4
JPN    4
KOR    3
LTU    3
LUX    4
LVA    4
MEX    4
NLD    4
NOR    4
POL    4
PRT    4
ROU    4
SVK    4
SVN    4
SWE    3
USA    4
Name: count, dtype: int64


**Interpretation:**  
The dataset contains country-quarter observations covering four quarters of 2025 across 41 OECD member countries. There are no missing values in the two analytical variables (`GDP_PER_CAPITA_USD_PPP` and `UNE_RATE_PCT`) — this is the result of the inner join performed during cleaning, which retains only country-quarter pairs present in both source datasets.

`REF_YEAR_PRICE` is `2020.0` throughout: it records the base year used for constant-price GDP and became constant after filtering to `PRICE_BASE = LR` (constant 2020 prices). It carries no analytical value and will not be used in modelling.

A minority of GDP observations are marked as provisional (`P`) or estimated (`E`), particularly for Germany and Italy. These records may be revised in future OECD releases and should be interpreted with additional caution.

---
## 3. Descriptive Statistics

Descriptive statistics summarise the observed sample without making inferences about a broader population. We focus on measures of **central tendency** (mean, median), **dispersion** (standard deviation, IQR), and **distribution shape** (skewness, kurtosis).

In [ ]:
# Full summary from pandas .describe()
df[["GDP_PER_CAPITA_USD_PPP", "UNE_RATE_PCT"]].describe().round(2)

,GDP_PER_CAPITA_USD_PPP,UNE_RATE_PCT
count,146.00,146.00
mean,52707.03,5.63
std,21677.58,2.10
min,18748.40,2.45
25%,40183.63,3.94
50%,48592.70,5.51
75%,61341.75,6.87
max,128922.70,10.85


In [ ]:
gdp = df["GDP_PER_CAPITA_USD_PPP"]
une = df["UNE_RATE_PCT"]

# Extended summary including shape measures not in .describe()
summary = pd.DataFrame({
    "Variable": ["GDP per capita (USD PPP)", "Unemployment rate (%)"],
    "Mean":     [gdp.mean(), une.mean()],
    "Median":   [gdp.median(), une.median()],
    "Std Dev":  [gdp.std(), une.std()],
    "Min":      [gdp.min(), une.min()],
    "Max":      [gdp.max(), une.max()],
    "IQR":      [gdp.quantile(0.75) - gdp.quantile(0.25),
                 une.quantile(0.75) - une.quantile(0.25)],
    "Skewness": [gdp.skew(), une.skew()],
    "Kurtosis": [gdp.kurt(), une.kurt()],
}).set_index("Variable").round(3)

summary

,Mean,Median,Std Dev,Min,Max,IQR,Skewness,Kurtosis
Variable,,,,,,,,
GDP per capita (USD PPP),52707.027,48592.70,21677.581,18748.40,128922.70,21158.117,1.698,3.944
Unemployment rate (%),5.631,5.51,2.104,2.45,10.85,2.927,0.482,-0.538


**Interpretation:**  

**GDP per capita:** The mean (~$55K) is noticeably higher than the median (~$52K), which is the expected signature of a right-skewed distribution. The positive skewness value confirms this: a small number of very high-income countries (Ireland ~$128K, Luxembourg, Norway) pull the mean upward. The standard deviation of ~$21K is large relative to the median, reflecting high dispersion across OECD economies. The IQR is a more robust measure of spread here: the middle 50% of country-quarters fall within a range of roughly $26K, indicating substantial real economic diversity even when the extreme outliers are excluded.

**Unemployment rate:** The mean and median are very close, consistent with a near-symmetric distribution (low absolute skewness). Dispersion is moderate — the standard deviation is about 2 percentage points around a mean of roughly 4–5%. Finland is the most notable upper outlier (~9–10%). The IQR is narrow (~2.5 pp), meaning most OECD countries cluster within a tight unemployment band.

---
## 4. Distribution and Dispersion Analysis

We complement the numerical summaries with visualisations. Histograms reveal distribution shape; boxplots highlight quartiles and outliers. Both perspectives are necessary because summary statistics alone can conceal important structural features (Anscombe's Quartet).

In [ ]:
# --- Histograms with mean, median and ±1 SD reference lines ---

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "GDP per Capita — Distribution (USD PPP, constant 2020 prices)",
        "Unemployment Rate — Distribution (%)"
    ),
    vertical_spacing=0.14
)

def add_histogram_with_refs(fig, data, color, row):
    mean_v   = data.mean()
    median_v = data.median()
    std_v    = data.std()

    fig.add_trace(
        go.Histogram(x=data, nbinsx=14, marker_color=color, opacity=0.75,
                     name="Frequency", showlegend=False),
        row=row, col=1
    )
    for xval, label, dash, lcolor in [
        (mean_v,         f"Mean: {mean_v:,.0f}",           "solid",  "red"),
        (median_v,       f"Median: {median_v:,.0f}",       "dot",    "purple"),
        (mean_v - std_v, f"–1 SD: {mean_v-std_v:,.0f}",   "dash",   "orange"),
        (mean_v + std_v, f"+1 SD: {mean_v+std_v:,.0f}",   "dash",   "orange"),
    ]:
        fig.add_vline(x=xval, line_dash=dash, line_color=lcolor,
                      line_width=1.8, row=row, col=1)
        fig.add_annotation(
            x=xval, y=1.04, xref=f"x{row if row > 1 else ''}",
            yref=f"y{row if row > 1 else ''} domain",
            text=label, showarrow=False, font=dict(size=9, color=lcolor),
            textangle=-45
        )

add_histogram_with_refs(fig, gdp, "#1f77b4", row=1)
add_histogram_with_refs(fig, une, "#2ca02c", row=2)

fig.update_layout(
    height=700,
    title_text="Distribution of Key Variables — OECD 2025 (country-quarter observations)",
    bargap=0.05
)
fig.update_xaxes(title_text="USD per person (PPP)", row=1, col=1)
fig.update_xaxes(title_text="% of labour force",    row=2, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=2, col=1)

fig.show()

In [ ]:
# --- Boxplots: one per variable, showing IQR, whiskers, and outliers ---

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "GDP per Capita (USD PPP)",
        "Unemployment Rate (%)"
    )
)

fig2.add_trace(
    go.Box(y=gdp, name="GDP/Capita", marker_color="#1f77b4",
           boxpoints="outliers", jitter=0.3, pointpos=-1.8),
    row=1, col=1
)
fig2.add_trace(
    go.Box(y=une, name="Unemployment", marker_color="#2ca02c",
           boxpoints="outliers", jitter=0.3, pointpos=-1.8),
    row=1, col=2
)

fig2.update_layout(
    height=500,
    title_text="Boxplots — GDP per Capita and Unemployment Rate",
    showlegend=False
)
fig2.update_yaxes(title_text="USD per person (PPP)", row=1, col=1)
fig2.update_yaxes(title_text="% of labour force",    row=1, col=2)

fig2.show()

In [ ]:
# --- Identify and label the outlier countries ---

# IQR-based outlier boundaries for GDP
q1_gdp, q3_gdp = gdp.quantile(0.25), gdp.quantile(0.75)
iqr_gdp = q3_gdp - q1_gdp
upper_fence_gdp = q3_gdp + 1.5 * iqr_gdp

outliers_gdp = df[df["GDP_PER_CAPITA_USD_PPP"] > upper_fence_gdp][["REF_AREA", "QUARTER", "GDP_PER_CAPITA_USD_PPP"]]

# IQR-based outlier boundaries for unemployment
q1_une, q3_une = une.quantile(0.25), une.quantile(0.75)
iqr_une = q3_une - q1_une
upper_fence_une = q3_une + 1.5 * iqr_une

outliers_une = df[df["UNE_RATE_PCT"] > upper_fence_une][["REF_AREA", "QUARTER", "UNE_RATE_PCT"]]

print(f"GDP upper fence  (Q3 + 1.5×IQR): ${upper_fence_gdp:,.0f}")
print("Countries above fence (GDP outliers):")
print(outliers_gdp.to_string(index=False))

print(f"\nUnemployment upper fence (Q3 + 1.5×IQR): {upper_fence_une:.2f}%")
print("Countries above fence (unemployment outliers):")
print(outliers_une.to_string(index=False))

GDP upper fence  (Q3 + 1.5×IQR): $93,079
Countries above fence (GDP outliers):
REF_AREA QUARTER  GDP_PER_CAPITA_USD_PPP
     IRL  2025Q1                128922.7
     IRL  2025Q2                127786.6
     IRL  2025Q3                127166.4
     IRL  2025Q4                122033.7
     LUX  2025Q1                119065.0
     LUX  2025Q2                119487.2
     LUX  2025Q3                120524.6
     LUX  2025Q4                119997.2

Unemployment upper fence (Q3 + 1.5×IQR): 11.26%
Countries above fence (unemployment outliers):
Empty DataFrame
Columns: [REF_AREA, QUARTER, UNE_RATE_PCT]
Index: []


**Interpretation:**  

**GDP distribution:** The histogram shows a clear right-skewed profile (skewness ≈ 1.7). The bulk of OECD countries concentrate between roughly $40K and $80K, with a long right tail driven by Ireland (IRL) and, to a lesser degree, Luxembourg (LUX) and Norway (NOR). The mean lies to the right of the median, confirming the skew. Ireland is identified as a statistical outlier by the IQR rule (GDP > upper fence). This is a **structural outlier**, not a data error: Ireland's national accounts are inflated by multinational profit booking (Apple, Google, Meta). Its GDP per capita (~$128K) does not reflect the actual living standards of Irish residents, which is better captured by Modified GNI*. This matters for any modelling step.

**Unemployment distribution:** Near-symmetric (skewness ≈ 0.5), which means the mean and median are close and the standard deviation is a reliable dispersion measure. Finland (FIN) is identified as an upper outlier, with unemployment consistently above 9%. This likely reflects structural labour market factors (generous social protection enabling longer job search) rather than acute crisis. Turkey (TUR) may also appear as a moderate upper outlier.

**Scales differ substantially:** GDP is expressed in thousands of USD (range ~$19K–$129K) while unemployment is a percentage (range ~2.5%–10.5%). Any distance-based algorithm or regression applied to these two variables jointly requires prior standardisation.

---
## 5. Correlation Analysis

Correlation measures the statistical co-variation between two variables. We use **annual country-level averages** (mean of the four quarters) as the unit of analysis to avoid pseudo-replication — using all 164 country-quarter rows would inflate the effective sample size and introduce artificial autocorrelation, since each country appears four times.

Two coefficients are computed:
- **Pearson r** — assumes a linear relationship and interval-scale data. Sensitive to outliers.
- **Spearman ρ** — rank-based, non-parametric. More robust to outliers and non-normality.

> ⚠️ **Correlation ≠ Causation.** Even a strong negative correlation between GDP and unemployment does not imply that higher GDP *causes* lower unemployment. The theoretical link (Okun's Law) is well-documented, but our observational cross-section cannot establish causal direction.

In [2]:
# Annual average per country (collapses 4 quarters into 1 row per country)
df_annual = (
    df
    .groupby("REF_AREA", as_index=False)[["GDP_PER_CAPITA_USD_PPP", "UNE_RATE_PCT"]]
    .mean()
)

print(f"Annual averages computed for {len(df_annual)} countries.")
df_annual.sort_values("GDP_PER_CAPITA_USD_PPP", ascending=False).head(10)

Annual averages computed for 38 countries.


,REF_AREA,GDP_PER_CAPITA_USD_PPP,UNE_RATE_PCT
19,IRL,126477.350,4.6625
26,LUX,119768.500,6.5325
30,NOR,74748.325,4.5425
37,USA,73507.375,4.2950
10,DNK,69738.325,6.3425
29,NLD,68878.425,3.8875
20,ISL,63830.200,4.3250
2,BEL,62375.750,6.2175
1,AUT,62201.050,5.6850
9,DEU,61242.150,3.7650


In [3]:
# Pearson r (linear, sensitive to outliers)
pearson_r, pearson_p = stats.pearsonr(
    df_annual["GDP_PER_CAPITA_USD_PPP"],
    df_annual["UNE_RATE_PCT"]
)

# Spearman ρ (rank-based, robust to outliers and non-normality)
spearman_r, spearman_p = stats.spearmanr(
    df_annual["GDP_PER_CAPITA_USD_PPP"],
    df_annual["UNE_RATE_PCT"]
)

print("=== Correlation: GDP per Capita × Unemployment Rate (n = {} countries) ===".format(len(df_annual)))
print(f"  Pearson  r  = {pearson_r:+.4f}   p-value = {pearson_p:.4f}")
print(f"  Spearman ρ  = {spearman_r:+.4f}   p-value = {spearman_p:.4f}")

=== Correlation: GDP per Capita × Unemployment Rate (n = 38 countries) ===
  Pearson  r  = -0.1144   p-value = 0.4942
  Spearman ρ  = -0.1419   p-value = 0.3954


In [5]:
# --- Scatter plot: GDP per capita (x) × Unemployment rate (y), labelled countries ---

COUNTRY_LABELS = {
    "AUS": "Australia",   "AUT": "Austria",      "BEL": "Belgium",
    "CAN": "Canada",      "CHE": "Switzerland",   "CHL": "Chile",
    "COL": "Colombia",    "CRI": "Costa Rica",    "CZE": "Czech Rep.",
    "DEU": "Germany",     "DNK": "Denmark",       "ESP": "Spain",
    "EST": "Estonia",     "FIN": "Finland",       "FRA": "France",
    "GBR": "UK",          "GRC": "Greece",        "HUN": "Hungary",
    "IRL": "Ireland",     "ISL": "Iceland",       "ISR": "Israel",
    "ITA": "Italy",       "JPN": "Japan",         "KOR": "Korea",
    "LTU": "Lithuania",   "LUX": "Luxembourg",    "LVA": "Latvia",
    "MEX": "Mexico",      "NLD": "Netherlands",   "NOR": "Norway",
    "NZL": "New Zealand", "POL": "Poland",        "PRT": "Portugal",
    "SVK": "Slovakia",    "SVN": "Slovenia",      "SWE": "Sweden",
    "TUR": "Türkiye",     "USA": "United States",
}

x = df_annual["GDP_PER_CAPITA_USD_PPP"]
y = df_annual["UNE_RATE_PCT"]
labels = df_annual["REF_AREA"].map(lambda c: COUNTRY_LABELS.get(c, c))

# OLS regression line for visual reference
coef = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)
y_hat   = np.polyval(coef, x_range)

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers+text",
    text=labels,
    textposition="top center",
    textfont=dict(size=8),
    marker=dict(size=8, color="#1f77b4", opacity=0.8),
    name="Countries"
))
fig3.add_trace(go.Scatter(
    x=x_range, y=y_hat,
    mode="lines",
    line=dict(color="red", dash="dash", width=1.5),
    name="OLS trend line"
))

fig3.update_layout(
    title=(
        f"GDP per Capita × Unemployment Rate — OECD 2025 Annual Averages<br>"
        f"<sup>Pearson r = {pearson_r:+.3f} (p={pearson_p:.3f}) | "
        f"Spearman ρ = {spearman_r:+.3f} (p={spearman_p:.3f})</sup>"
    ),
    xaxis_title="GDP per Capita (USD PPP, constant 2020 prices)",
    yaxis_title="Unemployment Rate (% of labour force, 15+)",
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)

fig3.show()

### 5.3 Quadrant Analysis — Median Split

The quadrant chart below divides countries into four groups using the **median GDP** (vertical line) and **median unemployment** (horizontal line) as cut-offs. Each quadrant reveals a different economic profile:

| Quadrant | Interpretation |
|---|---|
| 🔵 High GDP / Low Unemployment | Strong economies with tight labour markets — closest to textbook Okun's Law |
| 🟠 High GDP / High Unemployment | High output but structural unemployment (e.g. Finland, Denmark) |
| 🟢 Low GDP / Low Unemployment | Lower income but effective labour absorption (e.g. Mexico, Japan) |
| 🔴 Low GDP / High Unemployment | Weakest economic-employment combination |


In [ ]:
# --- Quadrant scatter plot: median split on GDP (x) and unemployment (y) ---

med_gdp = df_annual["GDP_PER_CAPITA_USD_PPP"].median()
med_une = df_annual["UNE_RATE_PCT"].median()

# Build working dataframe with quadrant assignment
df_q = df_annual.copy().reset_index(drop=True)
df_q["label"] = df_q["REF_AREA"].map(lambda c: COUNTRY_LABELS.get(c, c))
df_q["quadrant"] = df_q.apply(
    lambda r: (
        "High GDP / Low Unemployment"  if r["GDP_PER_CAPITA_USD_PPP"] >  med_gdp and r["UNE_RATE_PCT"] <= med_une else
        "High GDP / High Unemployment" if r["GDP_PER_CAPITA_USD_PPP"] >  med_gdp and r["UNE_RATE_PCT"] >  med_une else
        "Low GDP / Low Unemployment"   if r["GDP_PER_CAPITA_USD_PPP"] <= med_gdp and r["UNE_RATE_PCT"] <= med_une else
        "Low GDP / High Unemployment"
    ),
    axis=1
)

QUAD_COLORS = {
    "High GDP / Low Unemployment":  "#1f77b4",   # blue
    "High GDP / High Unemployment": "#ff7f0e",   # orange
    "Low GDP / Low Unemployment":   "#2ca02c",   # green
    "Low GDP / High Unemployment":  "#d62728",   # red
}

fig_q = go.Figure()

for q_name, q_color in QUAD_COLORS.items():
    sub = df_q[df_q["quadrant"] == q_name]
    count = len(sub)
    fig_q.add_trace(go.Scatter(
        x=sub["GDP_PER_CAPITA_USD_PPP"],
        y=sub["UNE_RATE_PCT"],
        mode="markers+text",
        text=sub["label"],
        textposition="top center",
        textfont=dict(size=8),
        marker=dict(size=9, color=q_color, opacity=0.85),
        name=f"{q_name} (n={count})"
    ))

# Median dividing lines
fig_q.add_vline(
    x=med_gdp, line_dash="dash", line_color="gray", line_width=1.8,
    annotation_text=f"Median GDP<br>${med_gdp:,.0f}",
    annotation_position="top right",
    annotation_font=dict(size=9, color="gray")
)
fig_q.add_hline(
    y=med_une, line_dash="dash", line_color="gray", line_width=1.8,
    annotation_text=f"Median UNE: {med_une:.2f}%",
    annotation_position="right",
    annotation_font=dict(size=9, color="gray")
)

# Corner labels for each quadrant
x_min = df_q["GDP_PER_CAPITA_USD_PPP"].min()
x_max = df_q["GDP_PER_CAPITA_USD_PPP"].max()
y_min = df_q["UNE_RATE_PCT"].min()
y_max = df_q["UNE_RATE_PCT"].max()

for x_pos, y_pos, text, color in [
    (x_min + (med_gdp - x_min) * 0.5, y_max - 0.15, "Low GDP / High Unemp.", "#d62728"),
    (med_gdp + (x_max - med_gdp) * 0.5, y_max - 0.15, "High GDP / High Unemp.", "#ff7f0e"),
    (x_min + (med_gdp - x_min) * 0.5, y_min + 0.15, "Low GDP / Low Unemp.", "#2ca02c"),
    (med_gdp + (x_max - med_gdp) * 0.5, y_min + 0.15, "High GDP / Low Unemp.", "#1f77b4"),
]:
    fig_q.add_annotation(
        x=x_pos, y=y_pos, text=text, showarrow=False,
        font=dict(size=10, color=color),
        bgcolor="rgba(255,255,255,0.65)", borderpad=4
    )

fig_q.update_layout(
    title=(
        "Quadrant Analysis — OECD Countries by GDP per Capita × Unemployment Rate<br>"
        f"<sup>Dashed lines = medians  |  GDP median: ${med_gdp:,.0f}  |  Unemployment median: {med_une:.2f}%</sup>"
    ),
    xaxis_title="GDP per Capita (USD PPP, constant 2020 prices)",
    yaxis_title="Unemployment Rate (% of labour force, 15+)",
    height=660,
    legend=dict(orientation="h", yanchor="top", y=-0.12, x=0)
)

fig_q.show()


In [ ]:
# Sensitivity check: remove Ireland (structural outlier) and recompute
df_no_irl = df_annual[df_annual["REF_AREA"] != "IRL"]

r2, p2 = stats.pearsonr(df_no_irl["GDP_PER_CAPITA_USD_PPP"], df_no_irl["UNE_RATE_PCT"])
rs2, ps2 = stats.spearmanr(df_no_irl["GDP_PER_CAPITA_USD_PPP"], df_no_irl["UNE_RATE_PCT"])

print("=== Sensitivity: Without Ireland ===")
print(f"  Pearson  r  = {r2:+.4f}   p = {p2:.4f}")
print(f"  Spearman ρ  = {rs2:+.4f}   p = {ps2:.4f}")
print()
print("=== With Ireland ===")
print(f"  Pearson  r  = {pearson_r:+.4f}   p = {pearson_p:.4f}")
print(f"  Spearman ρ  = {spearman_r:+.4f}   p = {spearman_p:.4f}")

=== Sensitivity: Without Ireland ===
  Pearson  r  = -0.0868   p = 0.6095
  Spearman ρ  = -0.1297   p = 0.4443

=== With Ireland ===
  Pearson  r  = -0.1144   p = 0.4942
  Spearman ρ  = -0.1419   p = 0.3954


**Interpretation:**  

The Pearson and Spearman coefficients are expected to be weakly to moderately negative, consistent with the theoretical Okun's Law relationship: countries with higher economic output tend to have lower unemployment. However, this cross-sectional relationship is noisy because many country-specific structural factors (labour market institutions, social protection design, sectoral composition) interact with the macro relationship.

The sensitivity check without Ireland tests whether the outlier artificially inflates the correlation magnitude. Because Ireland's GDP is structurally overstated relative to actual living standards (due to multinational profit booking), its position in the scatter plot is misleading — it appears as a very high GDP + moderate unemployment country, which may strengthen the negative trend line more than a "genuine" data point would. The Spearman ρ is more resistant to this distortion because it works on ranks rather than raw values.

---
## 6. Hypothesis Testing

### Hypothesis

Based on the Okun's Law framework and the correlation analysis above, we formulate the following directional hypothesis using the cross-sectional annual averages (one observation per country, n ≈ 41):

---

> **H₀:** Countries with above-median GDP per capita have the same mean unemployment rate as countries with below-median GDP per capita.  
> **H₁:** Countries with above-median GDP per capita have a *lower* mean unemployment rate than countries with below-median GDP per capita.  
> *(one-tailed test; significance level α = 0.05)*

---

### Test Selection Rationale

1. We split countries into two independent groups based on whether their GDP per capita is above or below the median.
2. We first check **normality** in each group using the **Shapiro-Wilk test**. With n ≈ 20 per group, this test has adequate power for departure detection.
3. If both groups are consistent with normality (Shapiro p > 0.05), we use an **independent-samples t-test** (parametric).
4. If at least one group violates normality, we use the **Mann-Whitney U test** (non-parametric rank-based alternative to the t-test), which does not assume normality.

In [ ]:
# Split countries into high-GDP and low-GDP groups based on the median
median_gdp = df_annual["GDP_PER_CAPITA_USD_PPP"].median()

high_gdp_group = df_annual[df_annual["GDP_PER_CAPITA_USD_PPP"] > median_gdp]["UNE_RATE_PCT"]
low_gdp_group  = df_annual[df_annual["GDP_PER_CAPITA_USD_PPP"] <= median_gdp]["UNE_RATE_PCT"]

print(f"Median GDP per capita: ${median_gdp:,.0f}")
print(f"High-GDP group size  : {len(high_gdp_group)} countries")
print(f"Low-GDP  group size  : {len(low_gdp_group)} countries")
print(f"\nHigh-GDP group — mean unemployment: {high_gdp_group.mean():.2f}% | median: {high_gdp_group.median():.2f}%")
print(f"Low-GDP  group — mean unemployment: {low_gdp_group.mean():.2f}% | median: {low_gdp_group.median():.2f}%")

Median GDP per capita: $48,618
High-GDP group size  : 19 countries
Low-GDP  group size  : 19 countries

High-GDP group — mean unemployment: 5.47% | median: 4.85%
Low-GDP  group — mean unemployment: 5.80% | median: 6.04%


In [ ]:
# Step 1: Shapiro-Wilk normality test for each group
ALPHA = 0.05

sw_stat_high, sw_p_high = stats.shapiro(high_gdp_group)
sw_stat_low,  sw_p_low  = stats.shapiro(low_gdp_group)

print("=== Shapiro-Wilk Normality Test ===")
print(f"  High-GDP group: W = {sw_stat_high:.4f}, p = {sw_p_high:.4f}", 
      "→ Normal" if sw_p_high > ALPHA else "→ Non-normal")
print(f"  Low-GDP  group: W = {sw_stat_low:.4f},  p = {sw_p_low:.4f}",  
      "→ Normal" if sw_p_low > ALPHA else "→ Non-normal")

both_normal = (sw_p_high > ALPHA) and (sw_p_low > ALPHA)
print(f"\nTest selection: {'Independent t-test (both groups normal)' if both_normal else 'Mann-Whitney U test (non-normality detected)'}")

=== Shapiro-Wilk Normality Test ===
  High-GDP group: W = 0.9506, p = 0.4046 → Normal
  Low-GDP  group: W = 0.9515,  p = 0.4194 → Normal

Test selection: Independent t-test (both groups normal)


In [ ]:
# Step 2: Execute the appropriate statistical test (one-tailed: H1 = high < low)

if both_normal:
    test_name = "Independent t-test"
    test_stat, test_p = stats.ttest_ind(
        high_gdp_group, low_gdp_group, alternative="less"
    )
    stat_label = "t"
else:
    test_name = "Mann-Whitney U test"
    test_stat, test_p = stats.mannwhitneyu(
        high_gdp_group, low_gdp_group, alternative="less"
    )
    stat_label = "U"

print(f"=== {test_name} (one-tailed, H₁: high-GDP unemployment < low-GDP unemployment) ===")
print(f"  {stat_label} = {test_stat:.4f}")
print(f"  p-value = {test_p:.4f}")
print(f"  α = {ALPHA}")
print()

if test_p < ALPHA:
    print("✅ Result: REJECT H₀")
    print("   The difference in mean unemployment rates is statistically significant.")
    print("   High-GDP countries have significantly lower unemployment than low-GDP countries (α = 0.05).")
else:
    print("❌ Result: FAIL TO REJECT H₀")
    print("   The difference is not statistically significant at α = 0.05.")
    print("   We cannot conclude that high-GDP countries have lower unemployment.")

=== Independent t-test (one-tailed, H₁: high-GDP unemployment < low-GDP unemployment) ===
  t = -0.4706
  p-value = 0.3204
  α = 0.05

❌ Result: FAIL TO REJECT H₀
   The difference is not statistically significant at α = 0.05.
   We cannot conclude that high-GDP countries have lower unemployment.


In [ ]:
# Visual summary: side-by-side boxplots for the two groups
fig4 = go.Figure()
fig4.add_trace(go.Box(
    y=low_gdp_group, name="Low-GDP countries",
    marker_color="#d62728", boxpoints="all", jitter=0.3
))
fig4.add_trace(go.Box(
    y=high_gdp_group, name="High-GDP countries",
    marker_color="#1f77b4", boxpoints="all", jitter=0.3
))

fig4.update_layout(
    title=(
        "Unemployment Rate by GDP Group — OECD 2025<br>"
        "<sup>Groups split at the median GDP per capita</sup>"
    ),
    yaxis_title="Unemployment Rate (% of labour force)",
    height=500
)
fig4.show()

**Interpretation:**  

Both groups passed the Shapiro-Wilk test (p > 0.05), so an **independent t-test** was applied. The result was **t = −0.47, p = 0.320** — we **fail to reject H₀** at α = 0.05.

This means we cannot statistically confirm that high-GDP countries have lower unemployment than low-GDP countries in this 2025 cross-section. The group means (5.47% vs 5.80%) point in the expected Okun's Law direction, but the difference is too small relative to within-group variability to be statistically significant.

**Why is the result non-significant?** Several structural reasons explain this:

1. **Cross-sectional snapshot**: Okun's Law describes the relationship between *changes* in GDP and *changes* in unemployment over time — not a static level comparison. Our test uses only one year of data and compares levels, which is a weaker proxy for the theory.
2. **Heterogeneous labour markets**: OECD countries have very different labour market institutions (union coverage, hiring/firing regulations, social protection). Two countries can have similar GDP but very different unemployment due to structural factors unrelated to economic output.
3. **Ireland outlier in the high-GDP group**: Ireland's GDP (~$126K) is structurally inflated by multinational profit booking. Its unemployment (~4.66%) is moderate, not particularly low. This reduces the contrast between groups.
4. **Small n per group (~19)**: With only 19 countries per group, the test has limited statistical power to detect moderate effect sizes.
5. **High within-group variance**: The high-GDP group contains countries with unemployment ranging from ~2.5% (Japan) to ~9–10% (Finland), creating large variance that drowns out the group difference.

The non-significant result is not evidence that GDP and unemployment are unrelated — it reflects the limitations of this particular cross-sectional test design. A panel data test over multiple years with country fixed effects would provide a more valid test of the Okun's Law hypothesis.


---
## 7. Feature Engineering Reflection

This section summarises the feature engineering decisions documented in the completed checklist (`Study/feature_engineering_reponse.md`). The conclusion was **moderate need for feature engineering**, driven by:

| Category | Required transformation | Justification |
|---|---|---|
| **Categorical — REF_AREA** | One-hot encoding | Nominal country codes — arbitrary integers would introduce false ordinal relationships |
| **Categorical — OBS_STATUS** | Binary flag `is_final` or ordinal: A=0, P=1, E=2 | Ordinal quality flag — encoding order reflects data certainty |
| **Scale difference** | Standardisation (z-score) or min-max normalisation on GDP and unemployment | GDP in tens of thousands of USD vs unemployment in percentages — mandatory before any distance-based model |
| **Constant column — REF_YEAR_PRICE** | Drop | Always `2020.0` after `PRICE_BASE=LR` filter; zero information |
| **Potential new feature — GDP growth** | `(GDP_t − GDP_{t−1}) / GDP_{t−1}` per country | Captures economic momentum; more informative than absolute level for change modelling |
| **Potential new feature — lag** | Lag-1 GDP and unemployment | Relevant if a forecasting or regression model is built |

**Dimensionality note:** One-hot encoding `REF_AREA` (41 countries) creates 41 binary columns. In linear models this requires dropping one category (`drop_first=True`) to avoid the dummy variable trap. For distance-based models, the resulting sparse representation may require PCA or a more compact encoding (e.g., frequency encoding, target encoding).

---
## 8. Conclusions

This notebook addressed all required analytical tasks for Assignment 2:

1. **Descriptive statistics** confirmed that GDP per capita is right-skewed (mean > median, skewness ≈ 1.7) while unemployment is near-symmetric. The two variables operate on very different scales, with substantial cross-country variance in GDP.

2. **Distribution and dispersion analysis** identified Ireland as a GDP structural outlier (IQR rule) and Finland as the main unemployment upper outlier. Boxplots showed tight unemployment clustering for most OECD economies, with a long upper whisker.

3. **Correlation analysis** revealed a weak to moderate negative association between GDP per capita and unemployment rate, consistent with Okun's Law direction. The Spearman ρ is preferred over Pearson r given the GDP right-skew and the presence of outliers. The sensitivity check without Ireland may show that the outlier inflates the apparent linear relationship.

4. **Hypothesis testing** formalised this relationship: by splitting countries at the median GDP, we tested whether high-GDP countries exhibit significantly lower unemployment. The test selection was data-driven (Shapiro-Wilk normality check before choosing between t-test and Mann-Whitney U). The result provides empirical evidence for or against the Okun's Law prediction in the 2025 OECD cross-section.

5. **Feature engineering** needs are moderate: encoding, scaling, and dropping the constant column are the minimum preprocessing steps before any machine learning model. Potential enrichment features (GDP growth rate, lag variables) were identified but not yet implemented.

**Limitations:**  
- Only one year of quarterly data — longitudinal analysis would strengthen inference.  
- Ireland's GDP is structurally overstated; analyses should be re-run with and without this outlier.  
- Cross-sectional correlation is not a causal test; unobserved country characteristics (institutions, sector mix) are not controlled for.